In [1]:
from qiskit_metal.quantrolib.chip import JAWS, Chiplet, Chip
from qiskit_metal.quantrolib.resonator import VariableIncaResonator, EdgeInductanceResonator, IncaResonator, BraggResonator
from qiskit_metal.quantrolib.simulation import ANSYS, RenderConfig, ReportConfig, EMSetup

#### Variable Declarations & Theory

In [2]:
wire_width = 2  # um
wire_length = 94  # um
real_wire_width = 0.6 # um

kinetic_inductance_square = 2.e-13  # H/square
inductance = kinetic_inductance_square * wire_width/real_wire_width * wire_length

nanowire_inductance = f"{inductance*1e12} pH"

## Sample designs

In [3]:
resonator_designs: dict[str, Chip] = {}

resonators = [
    # (IncaResonator, {"hfss_inductance": nanowire_inductance}),
    (BraggResonator, {

        "hfss_inductance": nanowire_inductance,
        "n_pairs": 18,
    }),
    # (VariableIncaResonator, {"hfss_inductance": nanowire_inductance}),
    # (EdgeInductanceResonator, {"hfss_inductance": nanowire_inductance}),
]

for resonator_type, options in resonators:
    name = resonator_type.__name__
    chip = Chiplet(size_x="4.8mm", size_y="2.4mm")
    # chip = JAWS()
    
    resonator = resonator_type(
        name=f"resonator_{name}",
        design=chip,
        options=options,
    )
    resonator_designs[name] = chip

### Simulation

In [ ]:
for name, chip in resonator_designs.items():

    chip._design.rebuild()

    print(chip._design.chips)
    
    render_config = RenderConfig(
        name="Renderer",
        project_name="Chiplet_resonator_frequency_only_test",
        design=chip._design,
        design_name=name,
        setups=[
            EMSetup(
                name="Setup",
                min_freq_ghz=4.0,
                n_modes=3,
                max_delta_f=0.1,
                max_passes=10,
                # max_passes=1,
            )
        ],
        open_pins=[],
        port_list=[],
        max_mesh_length_jj="0.5um",
        max_mesh_length_port="250um",
    )


    report_config = ReportConfig(
        name="Fields",
        field_configs=[
            {
                "object_name": "main",
                "QuantityName": 'Mag_E',
                'PlotFolder': 'E Field',
                "PlotGeomInfo_0": 1,
                "PlotGeomInfo_1": "Surface",
                "PlotGeomInfo_2": "FacesList",
                "PlotGeomInfo_3": 1,
                # "PlotGeomInfo_4": "12"  # Replace "12" with your desired face ID
            },
            {
                "object_name": "main",
                "QuantityName": 'Mag_H',
                'PlotFolder': 'H Field',
                "PlotGeomInfo_0": 1,
                "PlotGeomInfo_1": "Surface",
                "PlotGeomInfo_2": "FacesList",
                "PlotGeomInfo_3": 1,
                # "PlotGeomInfo_4": "12"  # Replace "12" with your desired face ID
            }
        ]
    )

    ansys = ANSYS(
        configs=[render_config, report_config],
        run_upon_init=False,
    )

    ansys.run_render()
    ansys.run_simulation()
    ansys.run_report()